# Chạy benchmark trên tập test

In [ ]:
import os
import numpy as np
from typing import Dict, Optional
from src.feature_extractor import FeatureExtractor
from src.detector import Detector
from src.gtalink import GTALink
from src.utils import run_pipeline
import warnings
warnings.filterwarnings("ignore")

In [ ]:
BASE = os.path.dirname(os.path.abspath(""))            # gốc repo (notebook nằm trong PipelineMOT/)

YOLO_PATH    = os.path.join(BASE, "PipelineMOT", "models", "best_yolo26.pt")
RF_PATH      = os.path.join(BASE, "PipelineMOT", "models", "best_rf_detr.pth")
SOLIDER_PATH = os.path.join(BASE, "PipelineMOT", "models", "swin_small_converted.pth")

VIDEOS_DIR      = os.path.join(BASE, "Data", "wide_view", "videos")
GROUNDTRUTH_DIR = os.path.join(BASE, "Data", "wide_view", "annotations")
ALL_VIDEOS      = sorted(os.listdir(VIDEOS_DIR))[-6:]  # test set: 1800–1980 s

OUTPUT_DIR = os.path.join(BASE, "Output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Hàm Testing

In [ ]:
def Testing(videos, detector, extractor,
            tracker, refiner, extractor_refiner,
            tracker_kwargs: Optional[Dict] = None,
            visualize=False, output_video=None,
            input_tracks_map=None,
            prev_time_accum=None,
            **viz_kwargs):

    all_results    = {}
    metrics_accum  = {}
    all_tracks_map = {}
    time_accum = {'normal': [], 'extract': [], 'gta': []}
    has_gta = refiner is not None

    det_name     = detector.backend if detector is not None else "None"
    ext_name     = extractor.backend if extractor is not None else "None"
    trk_name     = tracker if tracker is not None else "None"
    ref_ext_name = (extractor_refiner.backend if extractor_refiner is not None else
                    extractor.backend if extractor is not None else "None") \
                   if refiner is not None else "None"

    if refiner is not None:
        print(f"\n{'='*60}")
        print(f"PIPELINE: {det_name.upper()} -> {ext_name.upper()} -> "
              f"{trk_name.upper()} -> GTALink({ref_ext_name})")
        print(f"{'='*60}")
    else:
        print(f"\n{'='*60}")
        print(f"PIPELINE: {det_name.upper()} -> {ext_name.upper()} -> {trk_name.upper()}")
        print(f"{'='*60}")

    result_filename = f"result_{det_name}_{ext_name}_{trk_name}_{ref_ext_name}.txt"
    result_path     = os.path.join(OUTPUT_DIR, result_filename)

    for i, vid in enumerate(videos):
        video_path = os.path.join(VIDEOS_DIR, vid)
        gt_path    = os.path.join(GROUNDTRUTH_DIR, vid.split('.')[0] + ".csv")

        print(f"\n{'='*60}")
        print(f"Processing video: {vid}")
        print(f"{'='*60}")

        input_tracks = input_tracks_map.get(vid) if input_tracks_map is not None else None

        result = run_pipeline(
            video_path        = video_path,
            gt_csv_path       = gt_path,
            detector          = detector,
            tracker_name      = tracker,
            output_path       = output_video,
            extractor         = extractor,
            refiner           = refiner,
            refiner_extractor = extractor_refiner,
            tracker_kwargs    = tracker_kwargs or {},
            visualize         = visualize,
            input_tracks      = input_tracks,
            **viz_kwargs,
        )

        metrics = result['metrics']
        all_results[vid]    = metrics
        all_tracks_map[vid] = result['all_tracks']

        # ── Tích lũy timing ──────────────────────────────────────────
        if not has_gta:
            # Lần không GTA: lưu normal + extract thật, gta = 0
            time_accum['normal'].append(result['normal_time'])
            time_accum['extract'].append(result['extract_time'])
            time_accum['gta'].append(0.0)
        else:
            if prev_time_accum is not None:
                # Lần có GTA: lấy normal + extract từ lần chạy không GTA trước đó
                time_accum['normal'].append(prev_time_accum['normal'][i])
                time_accum['extract'].append(prev_time_accum['extract'][i])
            else:
                # Fallback: GTA chạy lại từ đầu (input_tracks_map=None)
                time_accum['normal'].append(result['normal_time'])
                time_accum['extract'].append(result['extract_time'])
            time_accum['gta'].append(result['gta_time'])

        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                metrics_accum.setdefault(k, []).append(v)

    avg_metrics = {k: np.mean(vals) for k, vals in metrics_accum.items()}

    n = len(videos)
    avg_normal  = np.sum(time_accum['normal'])  / n
    avg_extract = np.sum(time_accum['extract']) / n
    avg_gta     = np.sum(time_accum['gta'])     / n

    if has_gta:
        # normal + extract + gta chia 6
        avg_total = avg_normal + avg_extract + avg_gta
    else:
        # chỉ normal chia 6, KHÔNG cộng extract
        avg_total = avg_normal

    print(f"\n{'='*60}")
    print("AVERAGE METRICS ACROSS ALL VIDEOS:")
    for k, v in avg_metrics.items():
        print(f"  {k}: {v:.4f}")

    print(f"\nAVERAGE TIME PER VIDEO (across {n} videos):")
    if has_gta:
        print(f"  Normal time   : {avg_normal:.2f}s")
        print(f"  Extract time  : {avg_extract:.2f}s")
        print(f"  GTA time      : {avg_gta:.2f}s")
        print(f"  Total avg time: {avg_total:.2f}s")
    else:
        # Không hiện extract vì không tính vào tổng
        print(f"  Normal time   : {avg_normal:.2f}s")
        print(f"  Total avg time: {avg_total:.2f}s")
    print(f"{'='*60}")

    with open(result_path, 'w', encoding='utf-8') as f:
        f.write(f"Pipeline: {det_name} -> {ext_name} -> {trk_name} -> GTALink({ref_ext_name})\n")
        f.write(f"{'='*60}\n\n")
        for vid, metrics in all_results.items():
            f.write(f"[{vid}]\n")
            for k, v in metrics.items():
                f.write(f"  {k}: {v:.4f}\n" if isinstance(v, float) else f"  {k}: {v}\n")
            f.write("\n")
        f.write(f"{'='*60}\n")
        f.write("AVERAGE:\n")
        for k, v in avg_metrics.items():
            f.write(f"  {k}: {v:.4f}\n")

        f.write(f"\n{'='*60}\n")
        f.write(f"AVERAGE TIME PER VIDEO (across {n} videos):\n")
        if has_gta:
            f.write(f"  Normal time   : {avg_normal:.2f}s\n")
            f.write(f"  Extract time  : {avg_extract:.2f}s\n")
            f.write(f"  GTA time      : {avg_gta:.2f}s\n")
            f.write(f"  Total avg time: {avg_total:.2f}s\n")
        else:
            f.write(f"  Normal time   : {avg_normal:.2f}s\n")
            f.write(f"  Total avg time: {avg_total:.2f}s\n")

    print(f"\nKet qua da luu tai: {result_path}")

    return {
        'per_video'     : all_results,
        'average'       : avg_metrics,
        'all_tracks_map': all_tracks_map,
        'time_accum'    : time_accum,   # trả về để lần GTA dùng lại
    }

## Khởi tạo model

In [ ]:
detector_yolo  = Detector(backend="yolo26",  model_path=YOLO_PATH)
detector_rfdet = Detector(backend="rf_detr", model_path=RF_PATH)

extractor_color   = FeatureExtractor(backend="color_histogram")
extractor_osnet   = FeatureExtractor(backend="osnet")
extractor_solider = FeatureExtractor(
    backend="solider",
    solider_model_path=SOLIDER_PATH,
    solider_arch="swin_small",
    solider_semantic_weight=0.2,
)

refiner = GTALink(merge_dist_thres=0.35)

## Danh sách cấu hình

In [ ]:
BYTE   = {"use_project": False, "lambda_iou": 0.8}
OC_EMB = {"use_project": False, "use_emb": True}

CONFIGS = []
for det in ["yolo26", "rf_detr"]:
    for ext in ["osnet", "solider", "color_histogram"]:
        CONFIGS += [(det, ext, "bytetrack",  BYTE),
                    (det, ext, "strongsort", None),
                    (det, ext, "ocsort",     OC_EMB),
                    (det, ext, "deepeiou",   None)]
CONFIGS += [("yolo26",  None, "bytetrack", None),
            ("yolo26",  None, "ocsort",    None),
            ("rf_detr", None, "bytetrack", {"use_project": False, "lambda_iou": 1.0}),
            ("rf_detr", None, "ocsort",    {"use_project": False, "use_emb": False})]
print(len(CONFIGS), "online configs ×2 (± GTA)")

In [ ]:
DETECTORS  = {"yolo26": detector_yolo, "rf_detr": detector_rfdet}
EXTRACTORS = {None: None, "osnet": extractor_osnet, "solider": extractor_solider,
              "color_histogram": extractor_color}

def run_config(det, ext, trk, kwargs, videos=ALL_VIDEOS):
    r = Testing(videos, DETECTORS[det], EXTRACTORS[ext], trk,
                refiner=None, extractor_refiner=extractor_solider,
                input_tracks_map=None, tracker_kwargs=kwargs)
    Testing(videos, DETECTORS[det], EXTRACTORS[ext], trk,
            refiner=refiner, extractor_refiner=extractor_solider,
            input_tracks_map=r["all_tracks_map"], prev_time_accum=r["time_accum"],
            tracker_kwargs=kwargs)

## Chạy

In [ ]:
# Chọn cấu hình cần chạy, ví dụ chỉ pipeline tối ưu:
SELECT = [c for c in CONFIGS if c[:3] == ("yolo26", None, "bytetrack")]
# SELECT = CONFIGS                                   # toàn bộ lưới (nhiều giờ)

for det, ext, trk, kw in SELECT:
    run_config(det, ext, trk, kw)

## Kết quả đã chạy

In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join(BASE, "results", "mot_test_per_video.csv"), keep_default_na=False)
summary = (df.groupby(["detector", "extractor", "tracker", "gta"])
             [["MOTA", "IDF1", "HOTA", "IDs", "FP", "FN"]].mean().round(2))
summary